# Nemotron Local CV (Kaggle / RTX PRO 6000)

Runs `nemotron_local_cv.host_metric` against the 600-row stratified val split. Mirrors the host scorer's prompt + extraction + verification byte-for-byte.

Attach: this code dataset, the base model, and the trained adapter (either as another Dataset or as a kernel-source pointing at the training kernel's output).


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Untar our code bundle into /kaggle/working so the package layout is preserved.
import subprocess, sys
subprocess.run(
    ["tar", "xzf", f"/kaggle/input/nemotron-code/nemotron-code.tar.gz", "-C", "/kaggle/working"],
    check=True,
)
sys.path.insert(0, "/kaggle/working")

# Sanity: our packages import.
import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Apply pre-model-load Blackwell env. (post-load patches happen after model load.)
from kaggle.blackwell_setup import set_cuda_alloc_env, set_blackwell_ptxas_env  # noqa: E402
set_cuda_alloc_env()
set_blackwell_ptxas_env()
print("CUDA alloc env + Blackwell ptxas env set.")


In [ ]:
# vLLM + peft for inference. Kaggle's image may already have a compatible
# vllm — we pin the minimum that supports Nemotron MoE+Mamba LoRA loading.
import subprocess, sys
deps = [
    "vllm>=0.6.3",
    "peft>=0.13,<0.16",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *deps])


In [ ]:
import kagglehub
from pathlib import Path

BASE_MODEL_PATH = Path(kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"))
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")

DATA_DIR = Path("/kaggle/working/data")
VAL_SPLIT = DATA_DIR / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Locate the trained adapter. Two ways to attach it to this kernel:
#   (a) Add the training kernel as a kernel-source: adapter lives at
#       /kaggle/input/<train-kernel-slug>/adapter/
#   (b) Upload it as a private Dataset and attach: /kaggle/input/<dataset>/...
# Both code paths are covered by the glob below.
import glob

cands = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert cands, "No adapter_config.json found in /kaggle/input/**. Attach the trained-adapter source."
ADAPTER_DIR = str(Path(cands[0]).parent)
print(f"ADAPTER_DIR: {ADAPTER_DIR}")


In [ ]:
# Run local CV via the host-metric mirror.
import pandas as pd
import tempfile
from pathlib import Path
from nemotron_local_cv import host_metric

val_df = pd.read_csv(VAL_SPLIT)
print(f"val rows: {len(val_df)}")
print(val_df["label"].value_counts().to_dict())

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    val_df.to_csv(tmp_path / "test.csv", index=False)
    host_metric.configure(model_path=str(BASE_MODEL_PATH), data_path=tmp_path)

    solution = val_df[["id", "answer"]]
    submission_stub = pd.DataFrame([{"prediction": ADAPTER_DIR}])

    accuracy = host_metric.score(
        solution=solution,
        submission=submission_stub,
        row_id_column_name="id",
        debug=True,
    )

print(f"\n=== overall accuracy: {accuracy:.4f} ===")


In [ ]:
# Per-label breakdown joined from the debug CSV produced above.
from pathlib import Path

debug = Path("/kaggle/working/debug_predictions.csv")
if debug.exists():
    pred = pd.read_csv(debug, dtype="str").fillna({"extracted_prediction": "NOT_FOUND"})
    merge = val_df.astype({"id": "str"}).merge(pred.astype({"id": "str"}), on="id", how="left")
    merge["is_correct"] = merge.apply(
        lambda r: host_metric.verify(str(r.answer), str(r.extracted_prediction)),
        axis=1,
    )
    print(merge.groupby("label").is_correct.agg(["mean", "count"]).to_string())
    merge.to_csv("/kaggle/working/debug_predictions_with_labels.csv", index=False)
else:
    print("debug_predictions.csv not found — re-run with debug=True")
